# RAG Pipeline — Groq Edition

This notebook is identical to `rag_pipeline_walkthrough.ipynb` **except** it uses the **Groq API** instead of Ollama for LLM calls.

- Embeddings, ChromaDB, and Reranking are unchanged (they are local, no Ollama needed)
- Only the LLM (answer generation + evaluation judge) switches to Groq
- No existing source files are modified

**When to use this notebook:**
- Your machine has no GPU (CPU-only Ollama is too slow)
- You want faster evaluation runs (Groq is ~10x faster than CPU Ollama)

**Prerequisites:**
1. Free Groq API key from https://console.groq.com
2. `pip install langchain-groq`

---
## Step 0 — Install Groq SDK (run once)

In [ ]:
# Run this cell once to install the Groq LangChain integration
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'langchain-groq', '-q'])
print('Done.')

---
## Step 1 — Setup: Paths + Groq API Key

In [ ]:
import os, sys

# ── Path setup (same as walkthrough notebook) ──────────────────────────────
NOTEBOOK_DIR = os.path.abspath('')
REPO_ROOT    = os.path.dirname(NOTEBOOK_DIR)
SRC_PATH     = os.path.join(REPO_ROOT, 'src')
CHROMA_PATH  = os.path.join(REPO_ROOT, 'vectordb', 'chroma_db')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print('SRC_PATH   :', SRC_PATH)
print('CHROMA_PATH:', CHROMA_PATH)

# ── Groq API Key ────────────────────────────────────────────────────────────
# Option A: paste your key directly here (for local use only, don't commit)
GROQ_API_KEY = ""   # <── paste your key between the quotes

# Option B: read from environment variable (safer for shared repos)
# GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY is empty.\n"
        "Get a free key at https://console.groq.com → API Keys → Create key\n"
        "Then paste it into GROQ_API_KEY = \"...\" above."
    )

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print('\nGroq API key set.')

---
## Step 2 — Create the Groq LLM (replaces Ollama)

This is the **only difference** from the Ollama notebook.
`ChatGroq` is a drop-in replacement for `ChatOllama` — same interface, same LangChain methods.

In [ ]:
from langchain_groq import ChatGroq

# llama-3.1-8b-instant → same model family as Ollama's llama3.1:8b
# temperature=0 → deterministic (same question → same answer every time)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)

# Quick sanity check
test_response = llm.invoke("Reply with exactly: Groq is working.")
print(test_response.content)

---
## Step 3 — Load Dataset (RAGBench Finance)

Same as the walkthrough notebook — loads from HuggingFace.

In [ ]:
from datasets import load_dataset

ds = load_dataset("rungalileo/ragbench", "finqa", split="test")

print(f'Total rows : {len(ds)}')
print(f'Columns    : {ds.column_names}')

# Preview one sample
sample = ds[0]
print(f'\nQuestion   : {sample["question"]}')
print(f'Gold Answer: {sample["response"]}')

---
## Step 4 — Load Embedder

Local model — no Groq needed here. BAAI/bge-small-en-v1.5 runs on CPU.

In [ ]:
from embeddings.embedder import get_embedder

embedder = get_embedder()

# Quick test
test_vec = embedder.encode("rate of return on investment")
print(f'Embedding shape : {test_vec.shape}')
print(f'First 5 values  : {test_vec[:5].round(4)}')

---
## Step 5 — Connect to ChromaDB

Loads the **already-built** finance collection from disk.
No re-ingestion needed (uses the same vectordb built by the walkthrough notebook).

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Get existing collection — no embedding_function arg (we embed manually)
collection = client.get_collection(name="finance")

print(f'Collections available : {[c.name for c in client.list_collections()]}')
print(f'Finance chunk count   : {collection.count()}')

---
## Step 6 — Retrieve Chunks for a Query

In [ ]:
# Pick any question from the dataset — change the index to try different ones
sample = ds[0]
query  = sample['question']
gold   = sample['response']

print(f'Query      : {query}')
print(f'Gold Answer: {gold}')

# Embed the query
query_vector = embedder.encode(query).tolist()

# Retrieve top-10 chunks from ChromaDB
results = collection.query(
    query_embeddings=[query_vector],
    n_results=10
)

print(f'\nRetrieved {len(results["documents"][0])} chunks.')
print('\nTop-3 preview:')
for i, doc in enumerate(results['documents'][0][:3]):
    print(f'  [{i+1}] {doc[:120]}...')

---
## Step 7 — Rerank (local cross-encoder, no Groq)

In [ ]:
from reranking.reranker import rerank

reranked = rerank(query, results)[:3]

print(f'Top {len(reranked)} chunks after reranking:\n')
for i, chunk in enumerate(reranked):
    print(f'--- Chunk {i+1} ---')
    print(chunk[:200])
    print()

---
## Step 8 — Generate Answer with Groq LLM

Uses the same `ANSWER_PROMPT` from `src/prompts/answer_prompt.py` — no source file changed.

In [ ]:
from prompts.answer_prompt import ANSWER_PROMPT

def generate_answer_groq(query: str, context_chunks: list) -> str:
    """Same logic as src/llm/answer_generation.py but uses the Groq llm defined above."""
    context = "\n\n".join(context_chunks)
    prompt  = ANSWER_PROMPT.format(query=query, context=context)
    response = llm.invoke(prompt)
    return response.content.strip()

answer = generate_answer_groq(query, reranked)

print('Generated Answer:')
print(answer)
print(f'\nGold Answer: {gold}')

---
## Step 9 — Evaluate with Groq LLM as Judge

Uses the same `RAG_EVALUATION_PROMPT` and `RAGEvaluationCounts` schema — no source file changed.

In [ ]:
from prompts.evaluation_prompt import RAG_EVALUATION_PROMPT
from evaluation.evaluation_schema import RAGEvaluationCounts

def evaluate_groq(question: str, contexts: list, answer: str) -> RAGEvaluationCounts:
    """Same logic as src/evaluation/evaluation_metrics.py but uses the Groq llm."""
    prompt = RAG_EVALUATION_PROMPT.format(
        question=question,
        contexts="\n\n".join(contexts),
        answer=answer
    )
    judge = llm.with_structured_output(RAGEvaluationCounts)
    return judge.invoke(prompt)

def calculate_metrics(result: RAGEvaluationCounts) -> dict:
    return {
        "context_relevance":   result.relevant_chunks         / max(result.total_chunks, 1),
        "context_utilization": result.used_context_facts      / max(result.total_context_facts, 1),
        "completeness":        result.covered_relevant_facts  / max(result.total_relevant_facts, 1),
        "adherence":           result.supported_answer_statements / max(result.total_answer_statements, 1),
    }

eval_result = evaluate_groq(query, reranked, answer)
metrics     = calculate_metrics(eval_result)

print('RAG Evaluation Results (single question):')
print(f'  Context Relevance   : {metrics["context_relevance"]:.2f}')
print(f'  Context Utilization : {metrics["context_utilization"]:.2f}')
print(f'  Completeness        : {metrics["completeness"]:.2f}')
print(f'  Adherence           : {metrics["adherence"]:.2f}')
print(f'\nReasoning: {eval_result.reasoning}')

---
## Step 10 — Benchmark: Evaluate N Questions

Change `NUM_SAMPLES` to how many questions you want to evaluate.
With Groq this should finish in **2-4 minutes** for 10 questions.

In [ ]:
import time

NUM_SAMPLES = 10   # change to 20, 50, etc.

results_log = []
start_time  = time.time()

for i, row in enumerate(ds.select(range(NUM_SAMPLES))):
    q    = row['question']
    gold = row['response']
    print(f'[{i+1}/{NUM_SAMPLES}] {q[:70]}...')

    # Retrieve
    qvec     = embedder.encode(q).tolist()
    raw      = collection.query(query_embeddings=[qvec], n_results=10)
    chunks   = rerank(q, raw)[:3]

    # Generate
    ans      = generate_answer_groq(q, chunks)

    # Evaluate
    ev       = evaluate_groq(q, chunks, ans)
    m        = calculate_metrics(ev)
    m['question'] = q
    m['answer']   = ans
    m['gold']     = gold
    results_log.append(m)

    print(f'   CR:{m["context_relevance"]:.2f}  '
          f'CU:{m["context_utilization"]:.2f}  '
          f'CO:{m["completeness"]:.2f}  '
          f'AD:{m["adherence"]:.2f}')

elapsed = time.time() - start_time
print(f'\n{"="*55}')
print(f'AVERAGE OVER {NUM_SAMPLES} QUESTIONS  (took {elapsed:.0f}s total)')
print(f'{"="*55}')
for metric in ['context_relevance', 'context_utilization', 'completeness', 'adherence']:
    avg = sum(r[metric] for r in results_log) / len(results_log)
    print(f'  {metric:<25}: {avg:.4f}')

---
## Step 11 — Save Results to CSV (Optional)

Saves the per-question results to `benchmarks/groq_results.csv` so you can compare runs.

In [ ]:
import csv, os

out_dir  = os.path.join(REPO_ROOT, 'benchmarks')
out_file = os.path.join(out_dir, 'groq_results.csv')
os.makedirs(out_dir, exist_ok=True)

fields = ['question', 'gold', 'answer',
          'context_relevance', 'context_utilization', 'completeness', 'adherence']

with open(out_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for row in results_log:
        writer.writerow({k: row[k] for k in fields})

print(f'Saved {len(results_log)} rows → {out_file}')

---
## Quick Reference

| Component | What runs where |
|---|---|
| Embeddings (BAAI/bge-small-en-v1.5) | Local CPU |
| ChromaDB vector search | Local disk |
| Reranker (BAAI/bge-reranker-base) | Local CPU |
| LLM — Answer Generation | Groq API (cloud) |
| LLM — Evaluation Judge | Groq API (cloud) |

### Groq Free Tier Limits (as of mid-2025)
| Model | Requests/min | Tokens/min | Tokens/day |
|---|---|---|---|
| llama-3.1-8b-instant | 30 | 131,072 | 500,000 |

For 10 questions: each question uses ~2,000 tokens × 2 calls = ~40,000 tokens total → well within free limits.

### Switch to a Larger Model (Better Accuracy)
```python
# Replace llama-3.1-8b-instant with any supported Groq model:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)  # 70B — much smarter
llm = ChatGroq(model="llama-3.1-8b-instant",    temperature=0)  # 8B  — fast and free
```
Run `Step 2` again after changing the model name — no other cells need changing.